# Reto “Autoencoder como filtro de speckle”:
### Réplica del artículo, entrenamiento, ajuste de hiperparámetros, arquitectura Inception, métricas y análisis visual

**Artículo de referencia:** Cardona-Mesa et al., *"Optimization of Autoencoders for Speckle Reduction in SAR Imagery Through Variance Analysis and Quantitative Evaluation"*, Mathematics 2025, 13, 457.


## Puntos y dónde se resuelven

| # | Punto | Sección |
|---|---|---|
| 1 | Replicar la estructura del autoencoder según el artículo | 6 |
| 2 | Entrenar el autoencoder con una parte del dataset | 3 (`N_SUBSET`) y 7 |
| 3 | Ajustar hiperparámetros y/o estructura | 8 |
| 4 | Adaptar una arquitectura no secuencial (Inception) | 9 |
| 5 | Métricas ENL, SSIM, PSNR (y MSE) frente al ground truth | 10 |
| 6 | Validación visual: detalles, artefactos, pérdida de bordes | 11 |
| 7 | Evidencia, observaciones y conclusiones | 11 y 12 |

## Dataset: *SAR despeckling filters dataset*
Imágenes de Sentinel-1 (nivel 1 *Detected High-Res Dual-Pol*, GRD-HD, polarización VV) descargadas de ASF Data Search, sin georreferenciar. Se reescalaron a enteros de 8 bits, se registraron (alinearon) respecto a una imagen de referencia y se **promediaron 10 imágenes** (fusión multitemporal) para obtener la imagen **sin ruido (ground truth)**. Una de las 10 imágenes originales (la **ruidosa**) se divide en 1600 recortes de 512×512 px; lo mismo se hace con la imagen fusionada. Los recortes se separan en **1500 de entrenamiento y 100 de validación**. Todas las imágenes son TIFF `uint8` (0–255).

```
SAR despeckling filters dataset/
└── Main folder/
    ├── Noisy/        1500 imágenes con speckle (entrenamiento)
    ├── Noisy_val/     100 imágenes con speckle (validación)
    ├── GTruth/       1500 imágenes sin ruido   (entrenamiento)
    └── GTruth_val/    100 imágenes sin ruido   (validación)
```
La carpeta con las cuatro subcarpetas se **detecta automáticamente** a partir de `DATA_ROOT` (hasta 3 niveles). GT y Noisy se emparejan por nombre base; si los nombres no coinciden pero hay la misma cantidad, se emparejan por orden numérico y se muestran ejemplos para verificar.

**Consecuencia para la evaluación.** Como todos los recortes salen de la misma escena, un *split* aleatorio dentro de los 1500 de entrenamiento deja train y test muy parecidos. Por eso se evalúa en **dos conjuntos**: (a) **test interno** (15 % del subconjunto usado) y (b) **validación oficial** (`Noisy_val` / `GTruth_val`, 100 imágenes no usadas en el entrenamiento), que es la medida más fiable de generalización.

## Cómo ejecutarlo
1. Edita la sección **2. Configuración** (al menos `DATA_ROOT`).
2. Recomendado: primero ejecuta todo con `MODO_PRUEBA = True` (imágenes de 128 px, pocas épocas y pocos datos) para verificar el entorno en pocos minutos. Luego pon `False`, reinicia el kernel y ejecuta de nuevo (*Run All*).
3. Si ya entrenaste y guardaste los modelos (carpeta `modelos/`), pon `CARGAR_MODELOS = True` para **re-evaluarlos sin reentrenar** (tarda minutos en lugar de horas).


## 1. Instalación e importaciones

In [ ]:
# Si te faltan paquetes, descomenta y ejecuta UNA vez (luego reinicia el kernel):
# %pip install tensorflow opencv-python scikit-image scikit-learn scipy pandas matplotlib

import os, re, sys, json, time, random, itertools, warnings, platform, textwrap
from pathlib import Path

import numpy as np
import pandas as pd
import cv2
import matplotlib.pyplot as plt
from scipy.stats import wilcoxon
from sklearn.model_selection import train_test_split
from skimage.metrics import structural_similarity

warnings.filterwarnings("ignore")
pd.set_option("display.width", 220)
pd.set_option("display.max_columns", 60)

## 2. Configuración
Todo lo que normalmente querrás cambiar está en la celda siguiente.

| Parámetro | Efecto |
|---|---|
| `DATA_ROOT` | Carpeta local del dataset (la subcarpeta con `Noisy`, `Noisy_val`, `GTruth`, `GTruth_val` se busca sola) |
| `N_SUBSET` | Cuántos pares de los 1500 de entrenamiento se usan (la "parte del dataset"). Menos = más rápido |
| `N_VAL_OFICIAL` | Cuántas de las 100 imágenes de validación oficial se evalúan (100 = todas) |
| `IMG_SIZE` | 512 = tamaño nativo del dataset. Si lo cambias, las imágenes se re-muestrean (`INTER_AREA`), lo que promedia el speckle y altera sus estadísticas: úsalo solo si tu equipo no aguanta 512 |
| `EPOCHS_*` | Épocas de cada etapa |
| `SEARCH_SPACE`, `N_TRIALS` | Espacio y número de pruebas de la búsqueda de hiperparámetros |
| `INC_CFG` | Anchura de los módulos Inception |
| `CARGAR_MODELOS`, `BEST_CFG_MANUAL` | `True` = carga los modelos ya guardados en `MODEL_DIR` en lugar de entrenarlos (omite la búsqueda y usa `BEST_CFG_MANUAL` como configuración ajustada). Útil para re-evaluar sin reentrenar |

> **Memoria / tiempo:** 512×512 con 64 filtros y `batch_size` alto consume mucha VRAM. Si ves errores de memoria (*OOM*), baja `batch_size`, `filters` o `N_SUBSET`. La búsqueda captura el error y continúa con la siguiente configuración. En CPU, reduce `N_SUBSET`, las épocas y `N_TRIALS`.

In [ ]:
# ======================= CONFIGURACIÓN (edita aquí) =======================
MODO_PRUEBA = False   # True = corrida rápida de verificación (128x128, pocas épocas, pocos datos)

# --- Dataset LOCAL ---
DATA_ROOT     = Path("./SAR despeckling filters dataset")   
N_VAL_OFICIAL = 100   # imágenes de validación oficial a evaluar (máx. 100)

# --- Datos ---
IMG_SIZE  = 512
N_SUBSET  = 400     # pares usados de los 1500 de entrenamiento ("una parte del dataset")
TEST_FRAC = 0.15    # test interno: 15 % del subconjunto (como en el artículo, 85/15)
VAL_FRAC  = 0.10    # parte del 85 % restante reservada para validación / selección de modelos
SEED      = 123

# --- Modelo base: réplica de la Fig. 2 del artículo ---
BASE_CFG = dict(n_conv=4, filters=32, activation="relu", loss="bce", lr=1e-3, batch_size=4)

# --- Entrenamiento ---
EPOCHS_PRETRAIN = 5     # etapa 1: GT -> GT
EPOCHS_FINETUNE = 30    # etapa 2: Noisy -> GT (con early stopping)
EPOCHS_SEARCH   = 6     # épocas por modelo en la búsqueda de hiperparámetros

# --- Búsqueda de hiperparámetros (valores de la Tabla 1 del artículo + pérdida SSIM+L1) ---
SEARCH_SPACE = dict(
    n_conv=[4, 6], filters=[32, 64], activation=["relu", "elu"],
    loss=["mse", "bce", "ssim_l1"], lr=[1e-3, 5e-3, 1e-2], batch_size=[2, 4],
)
N_TRIALS = 8            # incluye siempre el modelo base como primera prueba

# --- Arquitectura Inception (punto 4): f filtros por rama; canales de salida del módulo = 4*f ---
INC_CFG = dict(f_enc=16, f_dec=8, activation="relu", n_down=2)

# --- ENL ---
ENL_WIN = 20            # ventana homogénea 20x20 (como en el artículo)
ENL_K   = 3             # nº de ventanas homogéneas por imagen

# --- Reutilizar modelos ya entrenados: solo evalúa, no reentrena ---
CARGAR_MODELOS  = True   # True = carga los .keras de MODEL_DIR (si existen) en lugar de entrenar; la búsqueda se omite
BEST_CFG_MANUAL = dict(n_conv=4, filters=64, activation="elu", loss="mse", lr=0.005, batch_size=2)   # solo con CARGAR_MODELOS=True: config. del modelo ajustado guardado

# --- Salida opcional: modelos entrenados ---
MODEL_DIR = Path("modelos")

if MODO_PRUEBA:
    IMG_SIZE, N_SUBSET, N_VAL_OFICIAL = 128, 30, 8
    EPOCHS_PRETRAIN, EPOCHS_FINETUNE, EPOCHS_SEARCH, N_TRIALS = 1, 2, 1, 2
    MODEL_DIR = Path("modelos_prueba")

random.seed(SEED); np.random.sehttps://data.mendeley.com/datasets/2xf5v5pwkr/2ed(SEED)
print("Dataset:", DATA_ROOT.resolve(), "| MODO_PRUEBA =", MODO_PRUEBA)

## 3. Datos: subconjunto de entrenamiento y validación oficial
- Se toma un subconjunto **aleatorio y reproducible** (`SEED`) de `N_SUBSET` pares de los 1500 de entrenamiento y se divide en **train**, **val** (10 % del 85 %, para *early stopping* y selección de hiperparámetros) y **test interno** (15 %). El test solo se usa para la evaluación final.
- Se cargan además las imágenes de **validación oficial** (`Noisy_val` / `GTruth_val`).
- Las imágenes se guardan como `uint8` y se normalizan a [0,1] al vuelo (`tf.data`), de modo que no se duplica el dataset en memoria.
- Para cada imagen se calculan las **ventanas homogéneas** del ENL sobre su ground truth.

In [ ]:
IMG_EXT = {".png", ".jpg", ".jpeg", ".tif", ".tiff", ".bmp"}
# Nombres de las 4 subcarpetas del dataset "SAR despeckling filters dataset"
REQ_DIRS = {"noisy": "Noisy", "noisy_val": "Noisy_val", "gt": "GTruth", "gt_val": "GTruth_val"}


def find_dataset_dirs(root, max_depth=3):
    # Busca (hasta max_depth niveles) la carpeta que contiene las 4 subcarpetas del dataset, sin distinguir mayúsculas.
    root = Path(root)
    if not root.exists():
        raise FileNotFoundError(f"No existe DATA_ROOT: {root.resolve()}\n  -> Edita DATA_ROOT en la celda de configuración.")
    base = len(root.resolve().parts)
    for cur, dirs, _ in os.walk(root):
        if len(Path(cur).resolve().parts) - base >= max_depth:
            dirs[:] = []
        subs = {d.lower(): Path(cur) / d for d in dirs}
        if all(v.lower() in subs for v in REQ_DIRS.values()):
            return {k: subs[v.lower()] for k, v in REQ_DIRS.items()}
    raise FileNotFoundError(f"No encontré las carpetas {list(REQ_DIRS.values())} dentro de {root.resolve()} (hasta {max_depth} niveles).")


def _natural_key(s):
    return [int(t) if t.isdigit() else t.lower() for t in re.split(r"(\d+)", s)]


def list_images(folder):
    return sorted((p for p in Path(folder).iterdir() if p.suffix.lower() in IMG_EXT), key=lambda p: _natural_key(p.name))


def read_gray(path, size):
    # Lee una imagen en escala de grises (8 bits). imdecode soporta rutas con tildes/espacios en Windows.
    buf = np.fromfile(str(path), dtype=np.uint8)
    img = cv2.imdecode(buf, cv2.IMREAD_GRAYSCALE)
    if img is None:
        raise IOError(f"No se pudo leer la imagen: {path}")
    if img.shape != (size, size):
        img = cv2.resize(img, (size, size), interpolation=cv2.INTER_AREA)
    return img


def pair_images(gt_dir, noisy_dir):
    # Empareja GT y Noisy: primero por nombre base; si no coinciden pero hay la misma cantidad, por orden numérico natural.
    gt_files, ny_files = list_images(gt_dir), list_images(noisy_dir)
    gt, ny = {p.stem: p for p in gt_files}, {p.stem: p for p in ny_files}
    common = sorted(set(gt) & set(ny), key=_natural_key)
    print(f"  {gt_dir.name}: {len(gt_files)} imgs | {noisy_dir.name}: {len(ny_files)} imgs | coinciden por nombre: {len(common)}")
    if common:
        return [(gt[s], ny[s]) for s in common]
    if len(gt_files) == len(ny_files) and gt_files:
        print("  AVISO: los nombres no coinciden; se emparejan por orden numérico. Verifica estos ejemplos:")
        for g, n in list(zip(gt_files, ny_files))[:3]:
            print(f"     {g.name}  <->  {n.name}")
        return list(zip(gt_files, ny_files))
    raise RuntimeError(f"No se pudo emparejar {gt_dir.name} con {noisy_dir.name} ({len(gt_files)} vs {len(ny_files)} imágenes).")


def make_splits(pairs, n_subset, test_frac, val_frac, seed):
    # Subconjunto aleatorio reproducible dividido en train / val / test.
    rng = np.random.default_rng(seed)
    n = len(pairs)
    k = min(n_subset, n)
    sel = np.sort(rng.choice(n, size=k, replace=False))
    sub = [pairs[i] for i in sel]
    idx = np.arange(k)
    tr_val, te = train_test_split(idx, test_size=test_frac, random_state=seed)
    tr, va = train_test_split(tr_val, test_size=val_frac, random_state=seed)
    return {"train": [sub[i] for i in tr], "val": [sub[i] for i in va], "test": [sub[i] for i in te]}


def load_pairs(pairs, size):
    gt, ny = [], []
    for i, (g, n) in enumerate(pairs):
        gt.append(read_gray(g, size))
        ny.append(read_gray(n, size))
        if (i + 1) % 100 == 0:
            print(f"    leídas {i + 1}/{len(pairs)}")
    return np.stack(ny).astype(np.uint8), np.stack(gt).astype(np.uint8)


def find_homogeneous_rois(img, win=20, k=3, stride=4):
    # k ventanas win x win "homogéneas" (menor coef. de variación sigma/mu) en la imagen de referencia (GT).
    # Se EXCLUYEN zonas casi negras (mu < 30), SATURADAS (mu > 245) y CONSTANTES (sigma < 1): en ellas el ENL no tiene
    # sentido (varianza ~0 => ENL astronómico) y distorsionaría las estadísticas.
    f = img.astype(np.float64)
    ii, ii2 = cv2.integral(f), cv2.integral(f * f)

    def box(a):
        return a[win:, win:] - a[:-win, win:] - a[win:, :-win] + a[:-win, :-win]

    n = win * win
    mean = box(ii) / n
    var = np.maximum(box(ii2) / n - mean ** 2, 0.0)
    std = np.sqrt(var)
    cvm = std / np.maximum(mean, 1e-9)
    for lo, hi, smin in ((30.0, 245.0, 1.0), (5.0, 250.0, 0.5), (0.0, 256.0, 0.0)):   # criterios, de estricto a laxo
        c = cvm.copy()
        c[(mean < lo) | (mean > hi) | (std < smin)] = np.inf
        ys, xs = np.mgrid[0:c.shape[0]:stride, 0:c.shape[1]:stride]
        cand = c[ys, xs].ravel()
        ys, xs = ys.ravel(), xs.ravel()
        rois = []
        for o in np.argsort(cand):
            if not np.isfinite(cand[o]):
                break
            y, x = int(ys[o]), int(xs[o])
            if all(abs(y - ry) >= win or abs(x - rx) >= win for ry, rx in rois):
                rois.append((y, x))
            if len(rois) == k:
                break
        if rois:
            return rois
    return [(0, 0)]

In [ ]:
def prepare_data():
    dirs = find_dataset_dirs(DATA_ROOT)
    print("Carpetas del dataset detectadas:")
    for k, v in dirs.items():
        print(f"  {k:>9}: {v}")
    data = {}
    print("\nEntrenamiento (Noisy / GTruth):")
    splits = make_splits(pair_images(dirs["gt"], dirs["noisy"]), N_SUBSET, TEST_FRAC, VAL_FRAC, SEED)
    for k, pr in splits.items():
        print(f"Cargando '{k}' ({len(pr)} pares) ...")
        ny, gt = load_pairs(pr, IMG_SIZE)
        data[k] = dict(noisy=ny, gt=gt, names=[g.stem for g, _ in pr])
    print("\nValidación oficial (Noisy_val / GTruth_val):")
    pv = pair_images(dirs["gt_val"], dirs["noisy_val"])
    if N_VAL_OFICIAL and N_VAL_OFICIAL < len(pv):
        sel = np.sort(np.random.default_rng(SEED).choice(len(pv), size=N_VAL_OFICIAL, replace=False))
        pv = [pv[i] for i in sel]
    print(f"Cargando 'oficial' ({len(pv)} pares) ...")
    ny, gt = load_pairs(pv, IMG_SIZE)
    data["oficial"] = dict(noisy=ny, gt=gt, names=[g.stem for g, _ in pv])
    for d in data.values():
        d["rois"] = [find_homogeneous_rois(g, ENL_WIN, ENL_K) for g in d["gt"]]
    return data

In [ ]:
data = prepare_data()
print()
for k, d in data.items():
    print(f"{k:>8}: {len(d['gt']):4d} imágenes | {d['gt'].shape[1:]} | uint8")

## 4. Métricas y funciones de visualización
Las métricas se calculan **por imagen** frente al ground truth y se reportan como media ± desviación.

| Métrica | Qué mide | Mejor |
|---|---|---|
| **MSE** | error cuadrático medio (0–255) | menor |
| **PSNR** | $10\log_{10}(255^2/\text{MSE})$ | mayor |
| **SSIM** | similitud estructural (luminancia, contraste, estructura) | → 1 |
| **ENL** | $\mu^2/\sigma^2$ en regiones homogéneas | **ENL / ENL_GT → 1** |

**Sobre el ENL.** El artículo lo trata como "mayor es mejor", pero un modelo que sobre-suaviza o colapsa a una imagen casi plana obtiene un ENL enorme sin ser mejor. Por eso se reporta el ENL del modelo **junto al del GT**, la razón `ENL / ENL_GT` (ideal ≈ 1) y, para la selección de modelos, `dENL = |ln(ENL / ENL_GT)|` (menor es mejor; invariante de escala). Las ventanas se eligen por ser las más homogéneas del GT, así que `ENL_GT` es una referencia optimista: interpreta la razón como una medida *relativa* entre modelos más que como un valor absoluto. **Se excluyen las ventanas casi negras (μ < 30), saturadas (μ > 245) y constantes (σ < 1)**, porque con varianza ≈ 0 el ENL se dispara (ENL → ∞) y distorsiona las estadísticas. Además el ENL tiene colas muy pesadas (unas pocas imágenes dominan la media), por eso las tablas reportan su **mediana** y el rango p25–p75.

**Diagnóstico complementario** (apoya el análisis visual; no está en el artículo):

| Indicador | Interpretación |
|---|---|
| **Ratio de gradientes** = gradiente medio (Sobel) de la salida / del GT | `< 0.8` suavizado excesivo o pérdida de bordes; `> 1.2` ruido residual o artefactos de alta frecuencia |
| **Correlación de bordes** entre los mapas de gradiente de la salida y del GT | mayor = bordes mejor ubicados |
| **Brillo pred/GT** | distinto de 1 = sesgo de brillo |
| **Contraste espectral** del error | cuánto sobresalen las frecuencias 1/4 y 1/2 del espectro del error frente a sus vecinas. `≈ 1–3` sin periodicidad; `> 6` hay una componente periódica detectable (heurística) |
| **Amplitud periódica estimada** (niveles de gris) | tamaño de esa componente. En imágenes de 512 px el contraste detecta componentes diminutas: `< 1` nivel de gris es imperceptible (el error típico es de ~15 niveles); `> 2` puede ser visible y es probable artefacto de tablero |

**Selección de modelos:** `paper_score` implementa el Algoritmo 1 del artículo (ranking por media + ranking por desviación estándar de cada métrica, ponderados; menor puntaje = mejor). El pseudocódigo del artículo calcula `RD` con `ranking(φ̄)`, que parece una errata; aquí el segundo ranking se hace sobre la **desviación**, como describe el texto.

In [ ]:
# ---------- Métricas (todas en escala 0-255, MAX = 255, como en el artículo) ----------
def grad_mag(img):
    g = img.astype(np.float32)
    gx = cv2.Sobel(g, cv2.CV_32F, 1, 0, ksize=3)
    gy = cv2.Sobel(g, cv2.CV_32F, 0, 1, ksize=3)
    return np.sqrt(gx ** 2 + gy ** 2)


def grad_corr(a, b):
    ga, gb = grad_mag(a).ravel(), grad_mag(b).ravel()
    if ga.std() < 1e-8 or gb.std() < 1e-8:
        return 0.0
    return float(np.corrcoef(ga, gb)[0, 1])


def enl_roi(img, roi, win=20):
    # ENL = mu^2 / sigma^2 en una región homogénea (Ec. 7 del artículo)
    y, x = roi
    p = img[y:y + win, x:x + win].astype(np.float64)
    return float(p.mean() ** 2 / max(p.var(), 1e-3))


def enl_img(img, rois, win=20):
    return float(np.mean([enl_roi(img, r, win) for r in rois]))


def periodic_component(err):
    # Detecta periodicidad en el error (típica de artefactos de tablero de Conv2DTranspose con stride 2).
    # Devuelve (contraste, amplitud):
    #   contraste = cuánto sobresale el espectro |FFT| en las frecuencias 1/4 y 1/2 respecto a sus vecinos inmediatos
    #               (~1-3: sin estructura periódica; > 6: hay una componente periódica detectable).
    #   amplitud  = amplitud estimada de esa componente en NIVELES DE GRIS (permite saber si es visible).
    # Ojo: en imágenes grandes (512 px) el contraste detecta componentes diminutas (< 1 nivel de gris), invisibles
    # frente a un error típico de ~15 niveles; por eso se reporta también la amplitud.
    e = err.astype(np.float64)
    e = e - e.mean()
    F = np.abs(np.fft.fft2(e))
    H, W = F.shape
    best_c, best_a = 0.0, 0.0
    for a in (0.0, 0.25, 0.5):
        for b in (0.0, 0.25, 0.5):
            if a == 0.0 and b == 0.0:
                continue
            y, x = int(round(a * H)) % H, int(round(b * W)) % W
            patch = F[np.ix_([(y + d) % H for d in range(-3, 4)], [(x + d) % W for d in range(-3, 4)])].copy()
            centro = patch[3, 3]
            patch[3, 3] = np.nan
            vecinos = max(float(np.nanmedian(patch)), 1e-9)
            self_conj = (a in (0.0, 0.5)) and (b in (0.0, 0.5))      # bins reales (Nyquist): no se reparten en +f y -f
            amp = (1.0 if self_conj else 2.0) * max(centro - vecinos, 0.0) / (H * W)
            best_c, best_a = max(best_c, float(centro / vecinos)), max(best_a, float(amp))
    return best_c, best_a


def per_image_metrics(pred, gt, noisy, rois):
    # pred, gt, noisy: arrays (N,H,W) en 0-255. Devuelve un DataFrame con una fila por imagen.
    rows = []
    for p, g, n, r in zip(pred, gt, noisy, rois):
        p, g, n = p.astype(np.float32), g.astype(np.float32), n.astype(np.float32)
        mse = float(np.mean((p - g) ** 2))
        enl_p, enl_g = enl_img(p, r, ENL_WIN), enl_img(g, r, ENL_WIN)
        ratio = enl_p / max(enl_g, 1e-9)
        pc, pa = periodic_component(p - g)
        rows.append(dict(
            MSE=mse,
            SSIM=float(structural_similarity(g, p, data_range=255.0)),
            PSNR=float(10 * np.log10(255.0 ** 2 / max(mse, 1e-10))),
            ENL=enl_p, ENL_GT=enl_g, ENL_RATIO=ratio, dENL=abs(float(np.log(max(ratio, 1e-9)))),
            GRAD_CORR=grad_corr(g, p),
            GRAD_RATIO=float(grad_mag(p).mean() / max(grad_mag(g).mean(), 1e-9)),
            BRILLO=float(p.mean() / max(g.mean(), 1e-9)),
            PICO_ESPECTRAL=pc, AMP_PERIODICA=pa,
        ))
    return pd.DataFrame(rows)


def noisy_metrics(d):
    nz = d["noisy"].astype(np.float32)
    return per_image_metrics(nz, d["gt"], d["noisy"], d["rois"])


def fmt_table(dfs):
    # Tabla "media ± desviación" por modelo + fila de referencia del Ground Truth.
    rows = []
    for name, df in dfs.items():
        rows.append({
            "Modelo": name,
            "MSE ↓": f"{df.MSE.mean():.1f} ± {df.MSE.std():.1f}",
            "SSIM ↑": f"{df.SSIM.mean():.3f} ± {df.SSIM.std():.3f}",
            "PSNR (dB) ↑": f"{df.PSNR.mean():.2f} ± {df.PSNR.std():.2f}",
            "ENL (mediana)": f"{df.ENL.median():.1f}",
            "ENL / ENL_GT (mediana, →1)": f"{df.ENL_RATIO.median():.2f}  [p25–p75: {df.ENL_RATIO.quantile(.25):.2f}–{df.ENL_RATIO.quantile(.75):.2f}]",
        })
    ref = next(iter(dfs.values()))
    rows.append({"Modelo": "Ground truth (referencia)", "MSE ↓": "0", "SSIM ↑": "1", "PSNR (dB) ↑": "∞",
                 "ENL (mediana)": f"{ref.ENL_GT.median():.1f}", "ENL / ENL_GT (mediana, →1)": "1.00"})
    return pd.DataFrame(rows)


def diag_table(dfs):
    # Diagnóstico numérico de bordes, brillo y artefactos periódicos (apoya el análisis visual).
    rows = []
    for name, df in dfs.items():
        rows.append({
            "Modelo": name,
            "Ratio gradientes (→1)": f"{df.GRAD_RATIO.mean():.2f} ± {df.GRAD_RATIO.std():.2f}",
            "Corr. bordes vs GT ↑": f"{df.GRAD_CORR.mean():.3f} ± {df.GRAD_CORR.std():.3f}",
            "Brillo pred/GT (→1)": f"{df.BRILLO.mean():.3f} ± {df.BRILLO.std():.3f}",
            "Contraste espectral (mediana; <6 = sin periodicidad)": f"{df.PICO_ESPECTRAL.median():.1f}",
            "Amplitud periódica est. (niveles de gris, mediana | p90)": f"{df.AMP_PERIODICA.median():.2f} | {df.AMP_PERIODICA.quantile(.9):.2f}",
        })
    return pd.DataFrame(rows)


def paper_score(summ, weights=None):
    # Algoritmo 1 del artículo: ranking por media + ranking por desviación, ponderado (menor puntaje = mejor).
    # summ: DataFrame con columnas {M}_mean y {M}_std para M en MSE, SSIM, PSNR, dENL.
    # El artículo usa ENL "mayor es mejor"; aquí se usa dENL = |ln(ENL / ENL_GT)| (menor es mejor) para no premiar el sobresuavizado.
    lower_is_better = {"MSE": True, "SSIM": False, "PSNR": False, "dENL": True}
    w = weights or {m: 0.25 for m in lower_is_better}
    score = pd.Series(0.0, index=summ.index)
    for m, lower in lower_is_better.items():
        r_mean = summ[f"{m}_mean"].rank(ascending=lower, method="min")
        r_std = summ[f"{m}_std"].rank(ascending=True, method="min")
        score += w[m] * (r_mean + r_std)
    return score


def final_ranking(dfs, names):
    rows = []
    for n in names:
        r = {"Modelo": n}
        for m in ("MSE", "SSIM", "PSNR", "dENL"):
            r[f"{m}_mean"], r[f"{m}_std"] = dfs[n][m].mean(), dfs[n][m].std()
        rows.append(r)
    s = pd.DataFrame(rows).set_index("Modelo")
    s["Puntaje (menor = mejor)"] = paper_score(s)
    return s.sort_values("Puntaje (menor = mejor)")


def paired_tests(dfs, pairs):
    # Prueba de Wilcoxon pareada por imagen entre dos modelos.
    rows = []
    for a, b in pairs:
        for m, mejor in (("PSNR", "mayor"), ("SSIM", "mayor"), ("MSE", "menor"), ("dENL", "menor")):
            x, y = dfs[a][m].values, dfs[b][m].values
            try:
                p = float(wilcoxon(x, y).pvalue)
            except Exception:
                p = float("nan")
            diff = float(x.mean() - y.mean())
            ok = diff > 0 if mejor == "mayor" else diff < 0
            rows.append({"A  vs  B": f"{a}  vs  {b}", "Métrica": m, "Mejor si es": mejor, "Media A": round(float(x.mean()), 4),
                         "Media B": round(float(y.mean()), 4), "A − B": round(diff, 4),
                         "p (Wilcoxon)": round(p, 4) if p == p else "n/d", "¿A mejor que B?": "sí" if ok else "no"})
    return pd.DataFrame(rows)


def auto_reading(dfs, ref_key):
    # Lectura AUTOMÁTICA de los números (heurística; hay que confirmarla visualmente).
    ref, out = dfs[ref_key], []
    for name, df in dfs.items():
        if name == ref_key:
            continue
        dp, ds = df.PSNR.mean() - ref.PSNR.mean(), df.SSIM.mean() - ref.SSIM.mean()
        mse_red = 100 * (1 - df.MSE.mean() / ref.MSE.mean())
        gr, enl_f, enl_g = df.GRAD_RATIO.mean(), df.ENL.median(), df.ENL_GT.median()
        t = (f"- {name}: PSNR {dp:+.2f} dB, SSIM {ds:+.3f}, reducción de MSE {mse_red:.1f}% respecto a la imagen sin filtrar. "
             f"ENL (mediana) = {enl_f:.1f} (GT = {enl_g:.1f}). ")
        if gr < 0.8:
            t += "Ratio de gradientes < 0.8: indicio de suavizado excesivo / pérdida de bordes. "
        elif gr > 1.2:
            t += "Ratio de gradientes > 1.2: indicio de ruido residual o artefactos de alta frecuencia. "
        else:
            t += "Ratio de gradientes cercano a 1: nitidez de bordes comparable a la del GT. "
        if df.ENL_RATIO.median() > 1.5:
            t += "ENL muy superior al del GT: posible sobresuavizado. "
        elif df.ENL_RATIO.median() < 0.7:
            t += "ENL inferior al del GT: posible speckle residual en zonas homogéneas. "
        else:
            t += "ENL cercano al del GT. "
        if abs(df.BRILLO.mean() - 1) > 0.05:
            t += f"Sesgo de brillo de {100 * (df.BRILLO.mean() - 1):+.1f}%. "
        amp = df.AMP_PERIODICA.median()
        if amp > 2.0:
            t += f"Componente periódica de ~{amp:.1f} niveles de gris: probable artefacto de tablero, puede ser visible. "
        elif df.PICO_ESPECTRAL.median() > 6:
            t += f"Componente periódica detectable pero débil (~{amp:.2f} niveles de gris, por debajo del umbral visual). "
        out.append(t.strip())
    return "\n".join(out)

In [ ]:
def _psnr(a, b):
    mse = float(np.mean((a.astype(np.float32) - b.astype(np.float32)) ** 2))
    return 10 * np.log10(255.0 ** 2 / max(mse, 1e-10))


def find_detail_roi(gt, size=96, stride=8):
    # Zona (size x size) con más energía de gradiente en el GT: sirve para inspeccionar bordes y detalles.
    size = int(min(size, gt.shape[0] // 2))
    ii = cv2.integral(grad_mag(gt).astype(np.float64))
    s = ii[size:, size:] - ii[:-size, size:] - ii[size:, :-size] + ii[:-size, :-size]
    s = s[::stride, ::stride]
    iy, ix = np.unravel_index(np.argmax(s), s.shape)
    return int(iy * stride), int(ix * stride)


def show_samples(d, n=3):
    n = min(n, len(d["gt"]))
    fig, axs = plt.subplots(n, 3, figsize=(12, 4 * n), squeeze=False)
    for r in range(n):
        gt, ny = d["gt"][r], d["noisy"][r]
        axs[r, 0].imshow(ny, cmap="gray", vmin=0, vmax=255); axs[r, 0].set_title("Noisy (entrada)")
        axs[r, 1].imshow(gt, cmap="gray", vmin=0, vmax=255); axs[r, 1].set_title("Ground truth (rojo = ROIs del ENL)")
        for (y, x) in d["rois"][r]:
            axs[r, 1].add_patch(plt.Rectangle((x, y), ENL_WIN, ENL_WIN, fill=False, edgecolor="red", lw=1.5))
        zs = max(32, gt.shape[0] // 5)
        y0, x0 = find_detail_roi(gt, zs)
        axs[r, 2].imshow(gt[y0:y0 + zs, x0:x0 + zs], cmap="gray", vmin=0, vmax=255, interpolation="nearest")
        axs[r, 2].set_title("Zoom en zona con detalle (GT)")
        for a in axs[r]:
            a.axis("off")
    plt.tight_layout(); plt.show()


def show_comparison(idxs, preds, d):
    # Por imagen: fila 1 = imagen completa, fila 2 = zoom (bordes/detalle), fila 3 = |error| vs GT, fila 4 = mapa de bordes (Sobel)
    for i in idxs:
        gt = d["gt"][i].astype(np.float32)
        cols = [("Noisy", d["noisy"][i].astype(np.float32))] + [(k, v[i]) for k, v in preds.items()] + [("Ground truth", gt)]
        zs = max(32, gt.shape[0] // 5)
        y0, x0 = find_detail_roi(gt, zs)
        gmax = float(np.percentile(grad_mag(gt), 99))
        fig, axs = plt.subplots(4, len(cols), figsize=(3.4 * len(cols), 14), squeeze=False)
        for j, (nm, im) in enumerate(cols):
            ttl = nm if nm == "Ground truth" else f"{nm}\nPSNR={_psnr(im, gt):.2f} dB"
            axs[0, j].imshow(im, cmap="gray", vmin=0, vmax=255); axs[0, j].set_title(ttl, fontsize=9)
            axs[0, j].add_patch(plt.Rectangle((x0, y0), zs, zs, fill=False, edgecolor="yellow", lw=1.5))
            axs[1, j].imshow(im[y0:y0 + zs, x0:x0 + zs], cmap="gray", vmin=0, vmax=255, interpolation="nearest")
            axs[2, j].imshow(np.abs(im - gt), cmap="inferno", vmin=0, vmax=100)
            axs[3, j].imshow(grad_mag(im), cmap="magma", vmin=0, vmax=gmax)
        for r, lab in enumerate(["Imagen", "Zoom (amarillo)", "|Error| vs GT", "Bordes (Sobel)"]):
            axs[r, 0].set_ylabel(lab, fontsize=11)
        for a in axs.ravel():
            a.set_xticks([]); a.set_yticks([])
        fig.suptitle(f"Imagen {i} del conjunto evaluado", y=1.0)
        plt.tight_layout(); plt.show()


def show_ratio_fft(i, preds, d):
    # Imagen ratio (Noisy/Filtrada) y espectro del error: ayudan a detectar estructura residual y artefactos periódicos.
    ny, gt = d["noisy"][i].astype(np.float32), d["gt"][i].astype(np.float32)
    fig, axs = plt.subplots(2, len(preds), figsize=(4.0 * len(preds), 8.2), squeeze=False)
    for j, (nm, p) in enumerate(preds.items()):
        ratio = ny / np.maximum(p[i], 1.0)
        axs[0, j].imshow(ratio, cmap="gray", vmin=0, vmax=3)
        axs[0, j].set_title(f"{nm}\nRatio Noisy/Filtrada (media={ratio.mean():.2f})", fontsize=9)
        err = p[i] - gt
        F = np.fft.fftshift(np.abs(np.fft.fft2(err - err.mean())))
        axs[1, j].imshow(np.log1p(F), cmap="viridis")
        axs[1, j].set_title("Espectro |FFT| del error\n(puntos aislados regulares = artefactos)", fontsize=9)
    for a in axs.ravel():
        a.set_xticks([]); a.set_yticks([])
    plt.tight_layout(); plt.show()


def plot_metric_bars(dfs):
    names = list(dfs)
    specs = [("PSNR", "PSNR (dB) ↑", None, "mean"), ("SSIM", "SSIM ↑", None, "mean"),
             ("ENL_RATIO", "ENL / ENL_GT (mediana y p25–p75; línea = 1)", 1.0, "median"),
             ("GRAD_RATIO", "Ratio de gradientes (línea = 1)", 1.0, "mean")]
    fig, axs = plt.subplots(1, 4, figsize=(19, 4))
    for ax, (col, t, ref, agg) in zip(axs, specs):
        if agg == "mean":
            vals = [dfs[n][col].mean() for n in names]
            err = [dfs[n][col].std() for n in names]
        else:
            vals = [dfs[n][col].median() for n in names]
            err = [[v - dfs[n][col].quantile(.25) for v, n in zip(vals, names)],
                   [dfs[n][col].quantile(.75) - v for v, n in zip(vals, names)]]
        ax.bar(range(len(names)), vals, yerr=err, capsize=4)
        ax.set_xticks(range(len(names)))
        ax.set_xticklabels([textwrap.fill(n, 14) for n in names], fontsize=8)
        ax.set_title(t, fontsize=10)
        if ref is not None:
            ax.axhline(ref, color="k", ls="--")
    plt.tight_layout(); plt.show()


def plot_history(histories):
    # histories: {nombre: {"pre": {...}, "ft": {...}}}
    fig, axs = plt.subplots(1, len(histories), figsize=(5.5 * len(histories), 4), squeeze=False)
    for ax, (nm, h) in zip(axs[0], histories.items()):
        tr = list(h.get("pre", {}).get("loss", [])) + list(h["ft"]["loss"])
        va = list(h.get("pre", {}).get("val_loss", [])) + list(h["ft"]["val_loss"])
        ax.plot(tr, label="loss (train)"); ax.plot(va, label="loss (val)")
        n_pre = len(h.get("pre", {}).get("loss", []))
        if n_pre:
            ax.axvline(n_pre - 0.5, color="gray", ls=":", label="fin etapa 1 (GT→GT)")
        ax.set_title(nm, fontsize=10); ax.set_xlabel("época"); ax.legend(fontsize=8)
    plt.tight_layout(); plt.show()

### Vista rápida de los datos y punto de partida
Imagen ruidosa, ground truth con las ventanas homogéneas elegidas para el ENL (rojo) y un zoom de la zona con más detalle. Debajo, las métricas de la **imagen sin filtrar** frente al GT: es el punto de partida que cualquier filtro debe mejorar.

In [ ]:
EVAL_SETS = ["test", "oficial"]
SET_LABEL = {"test": "Test interno (15 % del subconjunto de entrenamiento)",
             "oficial": "Validación oficial (Noisy_val / GTruth_val)"}
K_MAIN = "oficial"          # conjunto usado para las figuras y comparaciones principales
LBL_NOISY, LBL_BASE, LBL_BEST, LBL_INC = "Noisy (sin filtrar)", "CDAE base (artículo)", "CDAE ajustado", "Autoencoder Inception"

show_samples(data[K_MAIN], n=3)

NOISY_DF = {k: noisy_metrics(data[k]) for k in EVAL_SETS}
for k in EVAL_SETS:
    print(f"\n=== {SET_LABEL[k]} ({len(data[k]['gt'])} imágenes): imagen sin filtrar vs ground truth ===")
    display(fmt_table({LBL_NOISY: NOISY_DF[k]}))

## 5. TensorFlow y utilidades de entrenamiento
Se activa TensorFlow y se definen: el *pipeline* `tf.data`, las pérdidas (`mse`, `bce` y la combinación `ssim_l1`), el **entrenamiento en dos etapas** del artículo (GT→GT y luego Noisy→GT) con *early stopping* y reducción de *learning rate*, y la evaluación. En regresión de imágenes no se usa la métrica `accuracy`; se monitoriza el PSNR. Los modelos se identifican con cuatro etiquetas fijas (**Noisy (sin filtrar)**, **CDAE base (artículo)**, **CDAE ajustado**, **Autoencoder Inception**) y sus resultados se acumulan en `RESULTS`, `HISTORY`, `MODELS` y `TRAIN_TIME` para compararlos juntos en la sección 10.

In [ ]:
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers

tf.random.set_seed(SEED)
gpus = tf.config.list_physical_devices("GPU")
for g in gpus:
    try:
        tf.config.experimental.set_memory_growth(g, True)
    except Exception:
        pass
print("TensorFlow:", tf.__version__, "| Keras:", keras.__version__, "| GPUs:", [g.name for g in gpus] or "ninguna (CPU)")

In [ ]:
# ---------- Dataset (tf.data) a partir de arrays uint8 (no se duplica el dataset en memoria) ----------
def make_ds(x, y=None, batch=4, shuffle=False):
    s = x.shape[1]
    spec = tf.TensorSpec(shape=(s, s, 1), dtype=tf.float32)

    def gen():
        idx = np.random.permutation(len(x)) if shuffle else np.arange(len(x))
        for i in idx:
            xi = (x[i].astype(np.float32) / 255.0)[..., None]
            if y is None:
                yield xi
            else:
                yield xi, (y[i].astype(np.float32) / 255.0)[..., None]

    ds = tf.data.Dataset.from_generator(gen, output_signature=spec if y is None else (spec, spec))
    return ds.batch(batch).prefetch(tf.data.AUTOTUNE)


# ---------- Pérdidas ----------
def ssim_l1_loss(y_true, y_pred):
    # 0.84*(1-SSIM) + 0.16*L1 (combinación habitual en restauración de imágenes)
    ssim = tf.image.ssim(y_true, y_pred, max_val=1.0)
    l1 = tf.reduce_mean(tf.abs(y_true - y_pred), axis=[1, 2, 3])
    return 0.84 * (1.0 - ssim) + 0.16 * l1


def psnr_metric(y_true, y_pred):
    return tf.image.psnr(y_true, y_pred, max_val=1.0)


def get_loss(name):
    return {"mse": "mse", "bce": "binary_crossentropy", "ssim_l1": ssim_l1_loss}[name]


def compile_model(model, loss, lr):
    model.compile(optimizer=keras.optimizers.Adam(learning_rate=lr), loss=get_loss(loss), metrics=[psnr_metric])


# ---------- Entrenamiento en dos etapas (como en el artículo) ----------
def train_model(model, data, cfg, epochs_pre, epochs_ft, verbose=2):
    bs = cfg["batch_size"]
    hist = {"pre": {}, "ft": {}}
    if epochs_pre > 0:   # Etapa 1: GT -> GT (el modelo aprende la estructura de las imágenes SAR)
        compile_model(model, cfg["loss"], cfg["lr"])
        h = model.fit(make_ds(data["train"]["gt"], data["train"]["gt"], bs, shuffle=True),
                      validation_data=make_ds(data["val"]["gt"], data["val"]["gt"], bs),
                      epochs=epochs_pre, verbose=verbose)
        hist["pre"] = h.history
    # Etapa 2: Noisy -> GT (se reinicia el optimizador)
    compile_model(model, cfg["loss"], cfg["lr"])
    cbs = [keras.callbacks.EarlyStopping(monitor="val_loss", patience=8, restore_best_weights=True),
           keras.callbacks.ReduceLROnPlateau(monitor="val_loss", factor=0.5, patience=3, min_lr=1e-6)]
    h = model.fit(make_ds(data["train"]["noisy"], data["train"]["gt"], bs, shuffle=True),
                  validation_data=make_ds(data["val"]["noisy"], data["val"]["gt"], bs),
                  epochs=epochs_ft, callbacks=cbs, verbose=verbose)
    hist["ft"] = h.history
    return hist


def predict_255(model, x, batch=4):
    p = model.predict(make_ds(x, None, batch), verbose=0)
    return np.clip(p[..., 0] * 255.0, 0, 255).astype(np.float32)


def evaluate(model, d, batch=4):
    pred = predict_255(model, d["noisy"], batch)
    return pred, per_image_metrics(pred, d["gt"], d["noisy"], d["rois"])


def save_model(model, path):
    try:
        model.save(str(path))
        return True
    except Exception as e:
        print("No se pudo guardar el modelo completo:", e)
        return False

In [ ]:
RESULTS, HISTORY, MODELS, TRAIN_TIME = {}, {}, {}, {}   # se llenan a medida que se entrena cada modelo

def eval_and_store(name, model, batch):
    RESULTS[name] = {}
    for k in EVAL_SETS:
        pred, df = evaluate(model, data[k], batch)
        RESULTS[name][k] = dict(pred=pred, df=df)

def model_path(name):
    return MODEL_DIR / (name.split(" (")[0].replace(" ", "_").lower() + ".keras")

def try_load(name):
    # Con CARGAR_MODELOS=True y el archivo presente, devuelve el modelo guardado; si no, None (se entrenará)
    p = model_path(name)
    if CARGAR_MODELOS and p.exists():
        print(f"Cargando modelo guardado: {p}")
        return keras.models.load_model(str(p), compile=False)
    if CARGAR_MODELOS:
        print(f"AVISO: no existe {p}; este modelo se ENTRENARÁ (puede tardar).")
    return None

def fmt_min(x):
    return "—" if x is None else f"{x:.1f}"

def show_eval(names):
    for k in EVAL_SETS:
        print(f"\n=== {SET_LABEL[k]} ({len(data[k]['gt'])} imágenes) ===")
        display(fmt_table({LBL_NOISY: NOISY_DF[k], **{n: RESULTS[n][k]["df"] for n in names}}))

## 6. Punto 1 · Réplica de la estructura del autoencoder del artículo
**Arquitectura CDAE (Fig. 2 y Fig. 3 del artículo):**

```
Entrada 512×512×1
 ├─ ENCODER  (se repite n_conv/2 veces)
 │    Conv2D(F, 3×3, act, same) → MaxPool(2×2)        512×512×F → 256×256×F → 128×128×F  (latente)
 └─ DECODER  (se repite n_conv/2 veces)
      Conv2DTranspose(F, 3×3, strides=2, act, same)    128×128×F → 256×256×F → 512×512×F
 └─ SALIDA   Conv2D(1, 3×3, sigmoid)                    512×512×1
```
`n_conv ∈ {4, 6}`, `F ∈ {32, 64}` y `activation ∈ {relu, elu}` (Tabla 1). La celda comprueba que, con `n_conv=4` y `F=32`, las formas de cada capa coinciden con las de la Fig. 2.

In [ ]:
def build_cdae(img_size=512, n_conv=4, filters=32, activation="relu", name="CDAE"):
    assert n_conv % 2 == 0, "n_conv debe ser par (mitad encoder, mitad decoder)"
    n_half = n_conv // 2
    assert img_size % (2 ** n_half) == 0, f"img_size debe ser divisible por {2 ** n_half}"
    inp = layers.Input(shape=(img_size, img_size, 1), name="entrada")
    x = inp
    for i in range(n_half):                                   # ---- Encoder
        x = layers.Conv2D(filters, 3, padding="same", activation=activation, name=f"enc_conv{i + 1}")(x)
        x = layers.MaxPooling2D(2, padding="same", name=f"enc_pool{i + 1}")(x)
    for i in range(n_half):                                   # ---- Decoder
        x = layers.Conv2DTranspose(filters, 3, strides=2, padding="same", activation=activation, name=f"dec_deconv{i + 1}")(x)
    out = layers.Conv2D(1, 3, padding="same", activation="sigmoid", name="salida")(x)
    return keras.Model(inp, out, name=name)


# --- Verificación contra la Fig. 2 del artículo (n_conv=4, F=32) ---
m = build_cdae(IMG_SIZE, 4, 32, "relu")
S = IMG_SIZE
esperado = {"enc_conv1": (S, S, 32), "enc_pool1": (S // 2, S // 2, 32), "enc_conv2": (S // 2, S // 2, 32),
            "enc_pool2": (S // 4, S // 4, 32), "dec_deconv1": (S // 2, S // 2, 32), "dec_deconv2": (S, S, 32), "salida": (S, S, 1)}
try:
    for nombre, forma in esperado.items():
        obtenido = tuple(int(v) for v in m.get_layer(nombre).output.shape[1:])
        print(f"{nombre:12s} esperado {forma} | obtenido {obtenido} | {'OK' if obtenido == forma else 'DIFERENTE'}")
except Exception as e:
    print("No se pudo verificar automáticamente las formas (revisa el summary de abajo):", e)
m.summary()
del m

## 7. Punto 2 · Entrenamiento del modelo base con una parte del dataset
Se entrena la configuración base (`BASE_CFG`: 4 capas, 32 filtros, ReLU, BCE, lr 0.001, batch 4) con la receta de dos etapas del artículo, usando solo el subconjunto definido por `N_SUBSET`. Al terminar se evalúa en el test interno y en la validación oficial frente a la imagen sin filtrar y al GT.

In [ ]:
keras.backend.clear_session()
_m = try_load(LBL_BASE)
if _m is not None:
    MODELS[LBL_BASE], HISTORY[LBL_BASE], TRAIN_TIME[LBL_BASE] = _m, None, None
else:
    MODELS[LBL_BASE] = build_cdae(IMG_SIZE, BASE_CFG["n_conv"], BASE_CFG["filters"], BASE_CFG["activation"], name="CDAE_base")
    t0 = time.time()
    HISTORY[LBL_BASE] = train_model(MODELS[LBL_BASE], data, BASE_CFG, EPOCHS_PRETRAIN, EPOCHS_FINETUNE)
    TRAIN_TIME[LBL_BASE] = (time.time() - t0) / 60
print(f"\nModelo base | tiempo de entrenamiento: {fmt_min(TRAIN_TIME[LBL_BASE])} min | parámetros: {MODELS[LBL_BASE].count_params():,}")
eval_and_store(LBL_BASE, MODELS[LBL_BASE], BASE_CFG["batch_size"])
show_eval([LBL_BASE])

## 8. Punto 3 · Ajuste de hiperparámetros (y estructura)
Se prueba una muestra aleatoria del espacio de la Tabla 1 del artículo (más una pérdida `ssim_l1` y un `lr` de 0.01). Cada configuración se entrena `EPOCHS_SEARCH` épocas (etapa Noisy→GT) y se evalúa en el **split de validación interno** (no en el test ni en la validación oficial, para no filtrar información de evaluación a la selección). Los modelos se ordenan con el **Algoritmo 1 del artículo** (`paper_score`).

- El espacio completo (144 combinaciones) es demasiado costoso en un equipo local, por eso se muestrean `N_TRIALS` configuraciones; aumenta `N_TRIALS` si tienes GPU potente.
- La **estructura** (número de capas y filtros) forma parte de la búsqueda; la comparación con una estructura **no secuencial** se hace en la sección 9.
- En la búsqueda se omite la etapa 1 (GT→GT) por costo; la receta completa de dos etapas se aplica al entrenar el modelo final.

In [ ]:
def sample_trials(space, n, seed, include=None):
    keys = list(space)
    grid = [dict(zip(keys, g)) for g in itertools.product(*[space[k] for k in keys])]
    random.Random(seed).shuffle(grid)
    trials = [dict(include)] if include else []
    for cfg in grid:
        if cfg not in trials:
            trials.append(cfg)
        if len(trials) >= n:
            break
    return trials


def run_search():
    trials = sample_trials(SEARCH_SPACE, N_TRIALS, SEED, include=BASE_CFG)
    print(f"{len(trials)} configuraciones a probar (de {np.prod([len(v) for v in SEARCH_SPACE.values()])} posibles)\n")
    rows = []
    for t, cfg in enumerate(trials):
        print(f"[{t + 1}/{len(trials)}] {cfg}")
        keras.backend.clear_session()
        try:
            t0 = time.time()
            mdl = build_cdae(IMG_SIZE, cfg["n_conv"], cfg["filters"], cfg["activation"], name=f"trial_{t}")
            train_model(mdl, data, cfg, epochs_pre=0, epochs_ft=EPOCHS_SEARCH, verbose=0)
            _, dfv = evaluate(mdl, data["val"], cfg["batch_size"])
            row = dict(trial=t, **cfg, params=mdl.count_params(), time_s=round(time.time() - t0, 1))
            for mname in ("MSE", "SSIM", "PSNR", "ENL", "dENL", "GRAD_RATIO"):
                row[f"{mname}_mean"], row[f"{mname}_std"] = dfv[mname].mean(), dfv[mname].std()
            rows.append(row)
            print(f"      PSNR={row['PSNR_mean']:.2f} dB | SSIM={row['SSIM_mean']:.3f} | dENL={row['dENL_mean']:.3f} | {row['time_s']}s")
        except Exception as e:      # p. ej. falta de memoria (OOM) o lr inestable
            print("      -> configuración descartada:", type(e).__name__, str(e)[:120])

    search = pd.DataFrame(rows)
    assert len(search) > 0, "Ninguna configuración terminó. Reduce IMG_SIZE / batch_size / filters."
    search["score"] = paper_score(search)
    search = search.sort_values(["score", "PSNR_mean"], ascending=[True, False]).reset_index(drop=True)
    cols = ["trial", "n_conv", "filters", "activation", "loss", "lr", "batch_size", "params",
            "PSNR_mean", "SSIM_mean", "MSE_mean", "ENL_mean", "dENL_mean", "GRAD_RATIO_mean", "score", "time_s"]
    print("\nResultados de la búsqueda (split de validación interno), ordenados por el puntaje del artículo (menor = mejor):")
    display(search[cols].round(4))
    b = search.iloc[0]
    best = dict(n_conv=int(b.n_conv), filters=int(b.filters), activation=str(b.activation),
                loss=str(b.loss), lr=float(b.lr), batch_size=int(b.batch_size))
    return search, best


if CARGAR_MODELOS:
    best_cfg = dict(BEST_CFG_MANUAL)
    print("CARGAR_MODELOS=True: búsqueda omitida. Configuración ajustada tomada de BEST_CFG_MANUAL:")
else:
    search, best_cfg = run_search()
    print("\nMejor configuración:")
print(best_cfg)

### Entrenamiento final del modelo ajustado
Se entrena la mejor configuración con la receta completa de dos etapas y **las mismas épocas** que el modelo base, para que la comparación sea justa. Si la mejor configuración coincide con la base, se reutiliza el modelo base (y se indica).

In [ ]:
best_is_base = (best_cfg == BASE_CFG)
if best_is_base:
    print("La mejor configuración coincide con la base: se reutiliza el modelo base como 'CDAE ajustado'.")
    MODELS[LBL_BEST], HISTORY[LBL_BEST], TRAIN_TIME[LBL_BEST], RESULTS[LBL_BEST] = (
        MODELS[LBL_BASE], HISTORY[LBL_BASE], TRAIN_TIME[LBL_BASE], RESULTS[LBL_BASE])
else:
    keras.backend.clear_session()
    _m = try_load(LBL_BEST)
    if _m is not None:
        MODELS[LBL_BEST], HISTORY[LBL_BEST], TRAIN_TIME[LBL_BEST] = _m, None, None
    else:
        MODELS[LBL_BEST] = build_cdae(IMG_SIZE, best_cfg["n_conv"], best_cfg["filters"], best_cfg["activation"], name="CDAE_ajustado")
        t0 = time.time()
        HISTORY[LBL_BEST] = train_model(MODELS[LBL_BEST], data, best_cfg, EPOCHS_PRETRAIN, EPOCHS_FINETUNE)
        TRAIN_TIME[LBL_BEST] = (time.time() - t0) / 60
    eval_and_store(LBL_BEST, MODELS[LBL_BEST], best_cfg["batch_size"])
print(f"Parámetros: {MODELS[LBL_BEST].count_params():,} | tiempo de entrenamiento: {fmt_min(TRAIN_TIME[LBL_BEST])} min")
show_eval([LBL_BASE] if best_is_base else [LBL_BASE, LBL_BEST])

## 9. Punto 4 · Arquitectura no secuencial: autoencoder con módulos Inception
**Por qué Inception para speckle.** El speckle es ruido multiplicativo y espacialmente correlado: el "grano" tiene tamaños distintos y las estructuras útiles (bordes finos, parcelas, edificios) también. Un módulo Inception procesa la misma entrada con **varios tamaños de campo receptivo en paralelo** (1×1, 3×3, 5×5 y *pooling*) y concatena los resultados, de modo que la red elige la escala adecuada en cada zona. Es **no secuencial** porque el grafo tiene ramas paralelas que se vuelven a unir (`Concatenate`) en lugar de una simple cadena de capas.

**Cómo se adaptó.** Se conserva la **macroestructura del CDAE del artículo** (2 reducciones hasta 128×128, decoder con `Conv2DTranspose` de stride 2 y salida `Conv2D(1, sigmoid)`) y **cada `Conv2D` se sustituye por un módulo Inception**, de modo que la diferencia con el modelo secuencial se deba al tipo de bloque y no al resto del diseño. Se entrena con la misma receta y con la pérdida, `lr` y `batch_size` de la mejor configuración encontrada en la sección 8.

```
Módulo Inception (f filtros por rama, salida 4·f canales)
                 ┌── Conv 1×1 ───────────────────────┐
                 ├── Conv 1×1 → Conv 3×3 ────────────┤
   entrada ──────┼── Conv 1×1 (f/2) → Conv 5×5 ──────┼─► Concatenate ─► salida
                 └── MaxPool 3×3 (s=1) → Conv 1×1 ───┘

Autoencoder:  Entrada 512×512×1
   Inception(f_enc) → MaxPool → Inception(f_enc) → MaxPool            (latente 128×128×4·f_enc)
   ConvT(4·f_dec, s=2) → Inception(f_dec) → ConvT(4·f_dec, s=2) → Inception(f_dec)
   Conv2D(1, 3×3, sigmoid)                                             → 512×512×1
```
Las convoluciones 1×1 previas a las de 3×3 y 5×5 reducen canales (estilo Inception v1) para controlar el número de parámetros. El `summary()` muestra las ramas paralelas y las capas `Concatenate`.

In [ ]:
def inception_block(x, f, activation="relu", name="inc"):
    # Cuatro ramas paralelas con distinto campo receptivo; salida = 4*f canales
    b1 = layers.Conv2D(f, 1, padding="same", activation=activation, name=f"{name}_b1_1x1")(x)

    b2 = layers.Conv2D(f, 1, padding="same", activation=activation, name=f"{name}_b2_reduce")(x)
    b2 = layers.Conv2D(f, 3, padding="same", activation=activation, name=f"{name}_b2_3x3")(b2)

    b3 = layers.Conv2D(max(f // 2, 4), 1, padding="same", activation=activation, name=f"{name}_b3_reduce")(x)
    b3 = layers.Conv2D(f, 5, padding="same", activation=activation, name=f"{name}_b3_5x5")(b3)

    b4 = layers.MaxPooling2D(3, strides=1, padding="same", name=f"{name}_b4_pool")(x)
    b4 = layers.Conv2D(f, 1, padding="same", activation=activation, name=f"{name}_b4_proj")(b4)

    return layers.Concatenate(name=f"{name}_concat")([b1, b2, b3, b4])


def build_inception_ae(img_size=512, f_enc=16, f_dec=8, activation="relu", n_down=2, name="Inception_AE"):
    assert img_size % (2 ** n_down) == 0, f"img_size debe ser divisible por {2 ** n_down}"
    inp = layers.Input(shape=(img_size, img_size, 1), name="entrada")
    x = inp
    for i in range(n_down):                                   # ---- Encoder: Inception + MaxPool
        x = inception_block(x, f_enc, activation, f"enc_inc{i + 1}")
        x = layers.MaxPooling2D(2, padding="same", name=f"enc_pool{i + 1}")(x)
    for i in range(n_down):                                   # ---- Decoder: ConvT (x2) + Inception
        x = layers.Conv2DTranspose(4 * f_dec, 3, strides=2, padding="same", activation=activation, name=f"dec_deconv{i + 1}")(x)
        x = inception_block(x, f_dec, activation, f"dec_inc{i + 1}")
    out = layers.Conv2D(1, 3, padding="same", activation="sigmoid", name="salida")(x)
    return keras.Model(inp, out, name=name)


keras.backend.clear_session()
MODELS[LBL_INC] = build_inception_ae(IMG_SIZE, **INC_CFG)
MODELS[LBL_INC].summary()
n_concat = sum(isinstance(l, layers.Concatenate) for l in MODELS[LBL_INC].layers)
print(f"\nMódulos con ramas paralelas (Concatenate): {n_concat}  ->  arquitectura NO secuencial")

In [ ]:
TRAIN_CFG_INC = dict(loss=best_cfg["loss"], lr=best_cfg["lr"], batch_size=best_cfg["batch_size"])
_m = try_load(LBL_INC)
if _m is not None:
    MODELS[LBL_INC], HISTORY[LBL_INC], TRAIN_TIME[LBL_INC] = _m, None, None
else:
    print("Entrenando Inception con:", TRAIN_CFG_INC)
    t0 = time.time()
    HISTORY[LBL_INC] = train_model(MODELS[LBL_INC], data, TRAIN_CFG_INC, EPOCHS_PRETRAIN, EPOCHS_FINETUNE)
    TRAIN_TIME[LBL_INC] = (time.time() - t0) / 60
print(f"\nInception | tiempo de entrenamiento: {fmt_min(TRAIN_TIME[LBL_INC])} min | parámetros: {MODELS[LBL_INC].count_params():,}")
eval_and_store(LBL_INC, MODELS[LBL_INC], TRAIN_CFG_INC["batch_size"])
show_eval([LBL_INC])

## 10. Punto 5 · Métricas frente al ground truth: comparación final
Se comparan la imagen **sin filtrar**, el **modelo base del artículo**, el **modelo ajustado** y el **modelo Inception**, en los dos conjuntos de evaluación. Cada celda muestra:

1. La tabla de métricas (media ± desviación por imagen) con la fila de referencia del GT.
2. Las barras de PSNR, SSIM, ENL (línea = ENL del GT) y ratio de gradientes (línea = 1).
3. El **ranking final con el Algoritmo 1 del artículo** (menor puntaje = mejor).


In [ ]:
MODEL_NAMES = [LBL_BASE] + ([] if best_is_base else [LBL_BEST]) + [LBL_INC]
ALL_DFS = {k: {LBL_NOISY: NOISY_DF[k], **{n: RESULTS[n][k]["df"] for n in MODEL_NAMES}} for k in EVAL_SETS}

for k in EVAL_SETS:
    print(f"\n=============== {SET_LABEL[k]} ({len(data[k]['gt'])} imágenes) ===============")
    display(fmt_table(ALL_DFS[k]))
    plot_metric_bars(ALL_DFS[k])
    print("Ranking final (Algoritmo 1 del artículo; menor puntaje = mejor):")
    display(final_ranking(ALL_DFS[k], MODEL_NAMES).round(4))

## 11. Punto 6 · Validación visual y diagnóstico de artefactos
Cada figura muestra, para una imagen de la **validación oficial**: **imagen completa** (el recuadro amarillo marca la zona con más detalle del GT), **zoom** sobre esa zona, **mapa de |error| frente al GT** y **mapa de bordes (Sobel)**. Después se muestran la **imagen ratio** (Noisy / Filtrada) y el **espectro del error**. Debajo, una tabla de diagnóstico numérico (bordes, brillo, picos espectrales) y una lectura automática de los números.

### Guía de lectura
| Qué buscar | Cómo se ve | Dónde mirarlo |
|---|---|---|
| **Pérdida de bordes / borrosidad** | Bordes y líneas finas más anchos o ausentes; mapa de Sobel más apagado que el del GT | Filas "Zoom" y "Bordes"; ratio de gradientes < 0.8 |
| **Speckle residual** | Grano que persiste en zonas homogéneas | Fila "Zoom"; ENL por debajo del GT |
| **Sobresuavizado** | Zonas planas "tipo acuarela", textura aplanada | Fila "Zoom"; ENL muy superior al GT |
| **Artefactos de tablero (*checkerboard*)** | Cuadrícula regular, típica de `Conv2DTranspose` con `strides=2` | Fila "Zoom", mapa de error, puntos aislados en el espectro FFT, pico espectral > 6 |
| **Objetos o patrones falsos** | Estructuras en la salida que no están en el GT, o puntos brillantes eliminados | Fila "Zoom" y mapa de \|error\| |
| **Sesgo de brillo** | Salida globalmente más clara u oscura | Fila "Imagen"; brillo pred/GT ≠ 1; media del ratio ≠ 1 |

In [ ]:
rng = np.random.default_rng(SEED)
n_vis = min(3, len(data[K_MAIN]["gt"]))
VIS_IDX = sorted(rng.choice(len(data[K_MAIN]["gt"]), size=n_vis, replace=False).tolist())
preds_vis = {n: RESULTS[n][K_MAIN]["pred"] for n in MODEL_NAMES}

show_comparison(VIS_IDX, preds_vis, data[K_MAIN])
show_ratio_fft(VIS_IDX[0], preds_vis, data[K_MAIN])

In [ ]:
for k in EVAL_SETS:
    print(f"\n=== Diagnóstico de bordes, brillo y artefactos: {SET_LABEL[k]} ===")
    display(diag_table(ALL_DFS[k]))
    print("Lectura automática de los números (heurística):")
    print(auto_reading(ALL_DFS[k], LBL_NOISY))

### Observaciones visuales

- **Bordes y detalle.** Los tres modelos suavizan en exceso. Eliminan la textura fina que conserva el GT: la red de líneas de parcelas y caminos finos (imagen 1), la línea diagonal fina y la definición de los bloques brillantes (imagen 67) y las franjas de edificios de la zona urbana, que quedan como bloques borrosos (imagen 59). En los mapas de Sobel sobreviven solo los contornos grandes y anillos alrededor de puntos brillantes; el ratio de gradientes (≈ 0.56–0.64 frente a 1) lo confirma.
- **Speckle residual / suavizado.** No queda grano en las zonas homogéneas: la salida es más lisa que el propio GT, que conserva algo de grano porque es el promedio de solo 10 imágenes. El problema dominante es el **sobresuavizado**, no el speckle residual. En las imágenes ratio (Noisy / Filtrada) se distinguen estructuras reales (límites de parcelas, líneas): parte de la señal útil se eliminó junto con el ruido.
- **Artefactos o patrones extraños.** No se aprecian objetos falsos ni cuadrículas evidentes. Sí hay un **error sistemático en zonas saturadas (255)**: en la imagen 59 los modelos devuelven un gris claro en lugar de blanco puro, lo que deja un bloque de error casi constante. En el espectro del error no hay puntos aislados; la cruz horizontal/vertical que se ve es un efecto de borde de la FFT, no un artefacto. La tabla de diagnóstico cuantifica la componente periódica (amplitud estimada en niveles de gris): si es de ~1 nivel o menos, es imperceptible.
- **Diferencias entre modelos.** El base es el más borroso (pierde la línea diagonal fina y define peor los bloques brillantes de la imagen 67). El ajustado y el Inception conservan algo más de nitidez; entre ellos no hay una diferencia clara a simple vista (en el zoom de la imagen 1 el punto brillante inferior se distingue algo mejor en el Inception).
- **Modelo preferido.** Ajustado o Inception frente al base; entre ambos, empate visual. Coincide con las métricas (PSNR y MSE ligeramente a favor del Inception, SSIM a favor del ajustado).

## 12. Punto 7 · Conclusiones
La celda siguiente imprime un **resumen automático** a partir de los números (mejor modelo por métrica, mejora frente a la imagen sin filtrar y comparación entre el test interno y la validación oficial).

In [ ]:
for k in EVAL_SETS:
    dfs = ALL_DFS[k]
    rank = final_ranking(dfs, MODEL_NAMES)
    print(f"\n=============== {SET_LABEL[k]} ===============")
    print(f"Mejor modelo según el Algoritmo 1 del artículo: {rank.index[0]} (puntaje {rank.iloc[0]['Puntaje (menor = mejor)']:.2f})")
    for m, lower, etiqueta in (("PSNR", False, "PSNR (dB)"), ("SSIM", False, "SSIM"), ("MSE", True, "MSE"), ("dENL", True, "dENL = |ln(ENL/ENL_GT)|")):
        vals = {n: dfs[n][m].mean() for n in MODEL_NAMES}
        w = (min if lower else max)(vals, key=vals.get)
        print(f"  Mejor en {etiqueta}: {w} ({vals[w]:.4g}) | sin filtrar: {dfs[LBL_NOISY][m].mean():.4g}")
    print(auto_reading(dfs, LBL_NOISY))

print("\n=============== Generalización: test interno vs validación oficial ===============")
print(f"La imagen sin filtrar ya rinde distinto en ambos conjuntos: PSNR {ALL_DFS['test'][LBL_NOISY].PSNR.mean():.2f} dB (test) vs "
      f"{ALL_DFS['oficial'][LBL_NOISY].PSNR.mean():.2f} dB (oficial); por eso se compara la MEJORA sobre la imagen sin filtrar, no el PSNR absoluto.")
for n in MODEL_NAMES:
    g_in = ALL_DFS["test"][n].PSNR.mean() - ALL_DFS["test"][LBL_NOISY].PSNR.mean()
    g_of = ALL_DFS["oficial"][n].PSNR.mean() - ALL_DFS["oficial"][LBL_NOISY].PSNR.mean()
    nota = ("la mejora es claramente menor en la validación oficial: indicio de generalización limitada a otra zona de la escena"
            if g_in - g_of > 1.0 else "mejora similar en ambos conjuntos")
    print(f"- {n}: mejora de {g_in:+.2f} dB (test) vs {g_of:+.2f} dB (oficial) -> diferencia {g_in - g_of:+.2f} dB; {nota}.")

### Conclusiones

- **Replicación del artículo.** El CDAE base (4 capas, 32 filtros, ReLU, BCE, lr 0.001) reduce el MSE de 508 a 258, sube el PSNR de 21.34 a 24.66 dB y el SSIM de 0.569 a 0.638. Los modelos de la frontera de Pareto del artículo están en MSE 257–290, SSIM 0.65–0.70 y PSNR 23.8–24.4 dB: los resultados son de la misma magnitud (**comparables, no equivalentes**: aquí se entrenó con una parte del dataset y los protocolos de evaluación no son idénticos).
- **Ajuste de hiperparámetros.** Ganó 4 capas, 64 filtros, ELU, MSE, lr 0.005 y batch 2. Frente al base: +0.21 dB de PSNR, +0.023 de SSIM y MSE 257.6 → 240.8 (−6.5 %). Costo: 112 001 parámetros frente a 28 353 y 31.4 frente a 9.5 min de entrenamiento (CPU). La búsqueda (8 de 144 configuraciones, 6 épocas, una semilla) no distingue bien entre las mejores (≈ 0.4 dB de diferencia, al nivel del ruido), pero sí muestra dos cosas claras: las redes de **6 capas son más borrosas** (ratio de gradientes 0.27–0.44 frente a 0.43–0.71 en las de 4) y `lr = 0.01` con `ssim_l1` **diverge** (PSNR 10 dB).
- **Arquitectura Inception.** Con 47 457 parámetros (2.4 veces menos que el ajustado) iguala o supera al ajustado en PSNR y MSE (+0.10 dB; MSE 236.8 frente a 240.8) y es algo peor en SSIM (0.651 frente a 0.661). La inferencia es algo más lenta (43 frente a 38 ms por imagen) y el entrenamiento similar (27.6 frente a 31.4 min). La ventaja es de **eficiencia en parámetros**, no un salto de calidad.
- **Calidad visual.** Sobresuavizado y pérdida de bordes finos en los tres modelos; error sistemático en zonas saturadas; sin objetos falsos ni artefactos visibles (ver Observaciones visuales).
- **Generalización.** La mejora sobre la imagen sin filtrar es ≈ 1.4 dB menor en la validación oficial que en el test interno (base +4.73 → +3.33 dB; ajustado +4.95 → +3.53; Inception +4.97 → +3.64). El test interno es optimista porque comparte escena con el entrenamiento: usa la validación oficial como referencia.
- **Entrenamiento.** El *early stopping* no llegó a activarse en ninguno (35 épocas completas). En el ajustado y en el Inception (lr 0.005) hay picos de la loss al reiniciar el optimizador en la etapa 2 (en el ajustado el PSNR de validación cayó a 11.6 dB en la segunda época de esa etapa), de los que se recuperan.
- **Nota sobre el ENL.** En la primera corrida el ENL de la validación oficial quedó distorsionado: algunas imágenes tienen grandes zonas saturadas (255) con varianza ≈ 0 y el selector de ventanas las elegía. Ahora se excluyen (ver sección 4); con los modelos guardados basta poner `CARGAR_MODELOS = True` para re-evaluar. El ranking final (que usa `dENL`) puede cambiar ligeramente entre el ajustado y el Inception, cuyos puntajes estaban muy próximos (3.25 y 3.50).

## (Opcional) Guardar los modelos entrenados
Guarda los modelos en la carpeta `MODEL_DIR` para reutilizarlos sin volver a entrenar.

In [ ]:
if CARGAR_MODELOS:
    print("CARGAR_MODELOS=True: los modelos provienen de disco; no se vuelven a guardar.")
else:
    MODEL_DIR.mkdir(parents=True, exist_ok=True)
    for n in MODEL_NAMES:
        ok = save_model(MODELS[n], model_path(n))
        print(("guardado: " if ok else "NO guardado: ") + n)